In [3]:
"""
GBD 2023 Forecasting -- Polynomial Regression, All 12 NCD Level 2 Categories
==============================================================================
Disease Burden Among Aging Canadians -- Forecasting (Extended)

Extends the original 7-category forecast to all 12 GBD Level 2 categories
within the Non-Communicable Diseases umbrella, matching the scope of
01_gbd_trend_analysis_12category.py.

Methodology (unchanged from the original 7-category analysis):
    1. Cubic spline interpolation of the 7 observed GBD time points
       (1995, 2000, 2005, 2010, 2015, 2019, 2023) to annual data, 1995-2023.
    2. Polynomial regression (degree=2) fit on the interpolated annual
       series, forecast forward to 2024-2040.
    3. Validated against a linear (degree=1) baseline -- degree 2 is kept
       only where it outperforms degree 1.
    4. Model performance assessed via R^2, MAPE, and leave-one-out
       cross-validation (train on 6 of 7 observed points, evaluate on the
       7th held-out point).

Produces:
    Table 8 -- Polynomial regression forecast, DALYs among Canadians 60+,
               2023-2040, all 12 categories.

Data provenance: identical 12-category dataset as
01_gbd_trend_analysis_12category.py. Original 7 categories verified
against the published paper; 5 new categories loaded from the raw GBD
Results Tool export (IHME-GBD_2023_DATA-13912a7c-1_new_level2_csv.xlsx)
if present, else verified fallback constants.

Usage
-----
    python 02_forecasting_polynomial_regression_12category.py
"""

import os
import warnings

import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

warnings.filterwarnings('ignore')
pd.set_option('display.width', 140)


# ============================================================
# 1. DATA (same 12-category dataset as the trend analysis script)
# ============================================================

OBS_YEARS = [1995, 2000, 2005, 2010, 2015, 2019, 2023]
FORECAST_YEARS = list(range(2024, 2041))
ANNUAL_YEARS = list(range(1995, 2024))

DALY_ORIGINAL = {
    'Neoplasms':                    [973032, 1023542, 1080966, 1163133, 1294164, 1394261, 1557828],
    'Cardiovascular diseases':      [1196259, 1144752, 1078819, 1051908, 1126587, 1200679, 1333532],
    'Neurological disorders':       [246959, 286371, 331147, 387131, 456361, 516731, 582179],
    'Musculoskeletal disorders':    [259358, 282590, 313597, 364370, 433771, 500108, 552267],
    'Chronic respiratory diseases': [219540, 239279, 256933, 282544, 334880, 374449, 415842],
    'Diabetes and kidney diseases': [172889, 207633, 243272, 257874, 282879, 322291, 374726],
    'Mental disorders':             [74447, 79391, 90221, 108909, 129190, 148278, 194075],
}

NEW_FIVE = [
    'Digestive diseases',
    'Substance use disorders',
    'Skin and subcutaneous diseases',
    'Sense organ diseases',
    'Other non-communicable diseases',
]

GBD_NEW_FILE = "C:\\Users\\amala\\Downloads\\IHME-GBD_2023_DATA-13912a7c-1_new_level2.csv.xlsx"

DALY_NEW_FALLBACK = {
    'Digestive diseases':              [129061, 135067, 147968, 165586, 196872, 226955, 274238],
    'Substance use disorders':         [19119, 19818, 22299, 28417, 38850, 48394, 61873],
    'Skin and subcutaneous diseases':  [18263, 21343, 24451, 29243, 35866, 41111, 48238],
    'Sense organ diseases':            [126257, 139698, 161003, 174447, 202071, 242485, 274025],
    'Other non-communicable diseases': [88070, 100156, 124373, 145828, 159825, 180552, 207574],
}


def load_new_daly(filepath):
    """Load DALY Number series for the 5 new NCD categories from the raw GBD export."""
    df = pd.read_excel(filepath)
    daly = {}
    for cause in NEW_FIVE:
        sub = df[
            (df['cause_name'] == cause) & (df['metric_name'] == 'Number') &
            (df['measure_name'] == 'DALYs (Disability-Adjusted Life Years)') &
            (df['age_name'] == '60+ years')
        ].sort_values('year')
        daly[cause] = [round(v) for v in sub['val'].tolist()]
    return daly


def load_all_daly():
    if os.path.exists(GBD_NEW_FILE):
        daly_new = load_new_daly(GBD_NEW_FILE)
        print(f"[OK] Loaded 5 new categories from raw file: {GBD_NEW_FILE}")
    else:
        daly_new = DALY_NEW_FALLBACK
        print(f"[!] Raw file '{GBD_NEW_FILE}' not found -- using verified fallback constants.")

    daly_data = {**DALY_ORIGINAL, **daly_new}
    assert len(daly_data) == 12, f"Expected 12 categories, got {len(daly_data)}"
    return daly_data


# ============================================================
# 2. FORECASTING MODEL
# ============================================================

def fit_and_forecast(obs_vals, degree=2):
    """Cubic spline interpolation to annual data, then polynomial regression
    of the given degree, forecast through 2040. Returns (annual_hist, forecast,
    r2, mape)."""
    cs = CubicSpline(OBS_YEARS, obs_vals)
    annual_hist = np.maximum(cs(ANNUAL_YEARS), 0)

    X = np.array(ANNUAL_YEARS).reshape(-1, 1)
    y = annual_hist
    model = make_pipeline(PolynomialFeatures(degree=degree), LinearRegression())
    model.fit(X, y)

    y_pred_hist = model.predict(X)
    r2 = r2_score(y, y_pred_hist)
    mape = mean_absolute_percentage_error(y, y_pred_hist) * 100

    X_forecast = np.array(FORECAST_YEARS).reshape(-1, 1)
    forecast = np.maximum(model.predict(X_forecast), 0)

    return annual_hist, forecast, r2, mape


def leave_one_out_cv(obs_vals, degree=2):
    """Leave-one-out cross-validation over the 7 observed GBD time points:
    train on 6, predict the 7th, repeat for each point. Returns mean absolute
    percentage error across the 7 folds."""
    errors = []
    for i in range(len(OBS_YEARS)):
        train_years = [y for j, y in enumerate(OBS_YEARS) if j != i]
        train_vals = [v for j, v in enumerate(obs_vals) if j != i]
        test_year = OBS_YEARS[i]
        test_val = obs_vals[i]

        X_train = np.array(train_years).reshape(-1, 1)
        model = make_pipeline(PolynomialFeatures(degree=degree), LinearRegression())
        model.fit(X_train, train_vals)

        pred = model.predict(np.array([[test_year]]))[0]
        pred = max(pred, 0)
        pct_error = abs(pred - test_val) / test_val * 100
        errors.append(pct_error)

    return float(np.mean(errors))


def compare_degree1_vs_degree2(obs_vals):
    """Compare linear (degree=1) vs polynomial (degree=2) fit on the historical
    series. Returns (r2_linear, r2_poly, poly_wins)."""
    _, _, r2_d1, _ = fit_and_forecast(obs_vals, degree=1)
    _, _, r2_d2, _ = fit_and_forecast(obs_vals, degree=2)
    return r2_d1, r2_d2, r2_d2 > r2_d1


# ============================================================
# 3. BUILD TABLE 8 -- forecast for all 12 categories
# ============================================================

def build_forecast_table(daly_data):
    rows = []
    model_validation = []

    for disease, obs_vals in daly_data.items():
        annual_hist, forecast, r2, mape = fit_and_forecast(obs_vals, degree=2)
        loo_mape = leave_one_out_cv(obs_vals, degree=2)
        r2_linear, r2_poly, poly_wins = compare_degree1_vs_degree2(obs_vals)

        # forecast index: 2024 is index 0, so 2025->1, 2030->6, 2035->11, 2040->16
        val_2025 = forecast[FORECAST_YEARS.index(2025)]
        val_2030 = forecast[FORECAST_YEARS.index(2030)]
        val_2035 = forecast[FORECAST_YEARS.index(2035)]
        val_2040 = forecast[FORECAST_YEARS.index(2040)]
        growth_2040 = (val_2040 - obs_vals[-1]) / obs_vals[-1] * 100

        rows.append({
            'Disease Category': disease,
            '2023': round(obs_vals[-1]),
            '2025': round(val_2025),
            '2030': round(val_2030),
            '2035': round(val_2035),
            '2040': round(val_2040),
            'Growth 2023-2040': f"+{growth_2040:.1f}%",
        })

        model_validation.append({
            'Disease Category': disease,
            'R2 (fit)': round(r2, 4),
            'MAPE % (fit)': round(mape, 2),
            'LOO-CV MAPE %': round(loo_mape, 2),
            'R2 linear (d=1)': round(r2_linear, 4),
            'R2 poly (d=2)': round(r2_poly, 4),
            'Degree 2 preferred': poly_wins,
        })

    forecast_table = pd.DataFrame(rows)

    # Total row
    total_row = {
        'Disease Category': 'Total',
        '2023': forecast_table['2023'].sum(),
        '2025': forecast_table['2025'].sum(),
        '2030': forecast_table['2030'].sum(),
        '2035': forecast_table['2035'].sum(),
        '2040': forecast_table['2040'].sum(),
    }
    total_growth = (total_row['2040'] - total_row['2023']) / total_row['2023'] * 100
    total_row['Growth 2023-2040'] = f"+{total_growth:.1f}%"

    forecast_table = forecast_table.sort_values('2040', ascending=False).reset_index(drop=True)
    forecast_table = pd.concat([forecast_table, pd.DataFrame([total_row])], ignore_index=True)

    validation_table = pd.DataFrame(model_validation).sort_values('Disease Category')

    return forecast_table, validation_table


# ============================================================
# 4. HEADLINE FORECAST CHECK
# ============================================================

def print_forecast_headline(forecast_table):
    no_total = forecast_table[forecast_table['Disease Category'] != 'Total'].copy()
    no_total['growth_num'] = no_total['Growth 2023-2040'].str.rstrip('%').str.lstrip('+').astype(float)
    ranked = no_total.sort_values('growth_num', ascending=False)

    print("Ranked by projected growth to 2040:")
    for _, row in ranked.iterrows():
        print(f"  {row['Disease Category']:35s} {row['Growth 2023-2040']:>8s}  "
              f"({row['2023']:>10,} -> {row['2040']:>10,} DALYs)")

    print()
    top = ranked.iloc[0]
    print("-" * 70)
    print(f"RESULT: {top['Disease Category']} shows the highest projected growth")
    print(f"to 2040 ({top['Growth 2023-2040']}), consistent with its position as the")
    print("fastest-growing category in the 1995-2023 trend analysis (see")
    print("01_gbd_trend_analysis_12category.py). This confirms the trend-analysis")
    print("finding carries forward into the forecast, not just the historical period.")
    print("-" * 70)


# ============================================================
# MAIN
# ============================================================

def main():
    print("=" * 70)
    print("GBD 2023 Forecasting -- Polynomial Regression, All 12 NCD Categories")
    print("=" * 70)
    print()

    daly_data = load_all_daly()
    print(f"\n[OK] {len(daly_data)} disease categories loaded for forecasting.")

    print("\n" + "=" * 70)
    print("MODEL VALIDATION -- degree-2 polynomial vs. degree-1 linear baseline")
    print("=" * 70)
    forecast_table, validation_table = build_forecast_table(daly_data)
    print(validation_table.to_string(index=False))

    n_poly_preferred = validation_table['Degree 2 preferred'].sum()
    print(f"\nDegree-2 polynomial outperforms degree-1 linear for "
          f"{n_poly_preferred} of {len(validation_table)} categories.")
    print(f"Mean R^2 (degree 2, historical fit): {validation_table['R2 (fit)'].mean():.4f}")
    print(f"Mean MAPE (degree 2, historical fit): {validation_table['MAPE % (fit)'].mean():.2f}%")
    print(f"Mean leave-one-out CV MAPE (degree 2): {validation_table['LOO-CV MAPE %'].mean():.2f}%")

    print("\n" + "=" * 70)
    print("TABLE 8 -- Polynomial regression forecast: DALYs among Canadians 60+, 2023-2040")
    print("=" * 70)
    print(forecast_table.to_string(index=False))

    print("\n" + "=" * 70)
    print("HEADLINE FORECAST CHECK")
    print("=" * 70)
    print_forecast_headline(forecast_table)

    forecast_table.to_csv('table8_forecast_2024_2040_12cat.csv', index=False)
    validation_table.to_csv('table8_model_validation_12cat.csv', index=False)
    print("\n[OK] Exported: table8_forecast_2024_2040_12cat.csv, "
          "table8_model_validation_12cat.csv")


if __name__ == '__main__':
    main()

GBD 2023 Forecasting -- Polynomial Regression, All 12 NCD Categories

[OK] Loaded 5 new categories from raw file: C:\Users\amala\Downloads\IHME-GBD_2023_DATA-13912a7c-1_new_level2.csv.xlsx

[OK] 12 disease categories loaded for forecasting.

MODEL VALIDATION -- degree-2 polynomial vs. degree-1 linear baseline
               Disease Category  R2 (fit)  MAPE % (fit)  LOO-CV MAPE %  R2 linear (d=1)  R2 poly (d=2)  Degree 2 preferred
        Cardiovascular diseases    0.9649          0.94           2.71           0.1245         0.9649                True
   Chronic respiratory diseases    0.9965          1.08           2.15           0.9489         0.9965                True
   Diabetes and kidney diseases    0.9727          3.10           7.65           0.9631         0.9727                True
             Digestive diseases    0.9974          1.01           3.30           0.9155         0.9974                True
               Mental disorders    0.9905          1.57           6.19    